In [49]:
import pandas as pd
import numpy as np
import os

In [50]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


In [51]:
df = pd.read_csv(os.path.join(path,'IMDB Dataset.csv'))

In [52]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [53]:
df.shape

(50000, 2)

In [54]:
df = df.iloc[:10000]

In [55]:
df.shape

(10000, 2)

In [56]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [57]:
df['sentiment'].value_counts()

,count
sentiment,
positive,5028
negative,4972


In [58]:
df.isnull().sum()

,0
review,0
sentiment,0


In [59]:
df.duplicated().sum()

np.int64(17)

In [60]:
df.drop_duplicates(inplace=True)

In [61]:
df.duplicated().sum()

np.int64(0)

In [62]:
import re

def remove_tags(text):
  clean_text=re.sub(re.compile('<.*?>'),'',text)
  return clean_text

In [63]:
df['review']=df['review'].apply(remove_tags)

In [64]:
df['review']=df['review'].apply(lambda x:x.lower())

In [65]:
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')

sw_list=stopwords.words('english')
df['review'] = df['review'].apply(lambda x:[item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [66]:
X=df.iloc[:,0:1]
y=df['sentiment']

In [67]:
X

,review
0,one reviewers mentioned watching 1 oz episode ...
1,wonderful little production. filming technique...
2,thought wonderful way spend time hot summer we...
3,basically there's family little boy (jake) thi...
4,"petter mattei's ""love time money"" visually stu..."
...,...
9995,"fun, entertaining movie wwii german spy (julie..."
9996,"give break. anyone say ""good hockey movie""? kn..."
9997,movie bad movie. watching endless series bad h...
9998,"movie probably made entertain middle school, e..."


In [68]:
y

,sentiment
0,positive
1,positive
2,positive
3,negative
4,positive
...,...
9995,positive
9996,negative
9997,negative
9998,negative


In [69]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y = le.fit_transform(y)

In [70]:
y

array([1, 1, 1, ..., 0, 0, 1])

In [79]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)

In [80]:
X_train.shape

(7986, 1)

In [100]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
cv=CountVectorizer()

In [84]:
X_train_bow=cv.fit_transform(X_train['review']).toarray()
X_test_bow=cv.transform(X_test['review']).toarray()

In [85]:
X_train_bow.shape

(7986, 48282)

In [86]:
from sklearn.naive_bayes import GaussianNB

gnb=GaussianNB()

In [87]:
gnb.fit(X_train_bow,y_train)

GaussianNB()

In [90]:
y_pred=gnb.predict(X_test_bow)
from sklearn.metrics import accuracy_score,confusion_matrix
accuracy_score(y_test,y_pred)

0.6324486730095142

In [91]:
confusion_matrix(y_test,y_pred)

array([[717, 235],
       [499, 546]])

In [97]:
from sklearn.ensemble import RandomForestClassifier

cv=CountVectorizer(max_features=3000,ngram_range=(1,3))

X_train_bow=cv.fit_transform(X_train['review']).toarray()
X_test_bow=cv.transform(X_test['review']).toarray()

rf=RandomForestClassifier()
rf.fit(X_train_bow,y_train)
y_pred=rf.predict(X_test_bow)
accuracy_score(y_test,y_pred)

0.8417626439659489

In [99]:
confusion_matrix(y_test,y_pred)

array([[800, 152],
       [164, 881]])

# Using Tfidf

In [102]:
from sklearn.ensemble import RandomForestClassifier

tfidf=TfidfVectorizer(max_features=5000)

X_train_bow=tfidf.fit_transform(X_train['review']).toarray()
X_test_bow=tfidf.transform(X_test['review']).toarray()

rf=RandomForestClassifier()
rf.fit(X_train_bow,y_train)
y_pred=rf.predict(X_test_bow)
accuracy_score(y_test,y_pred)

0.8447671507260891

In [103]:
confusion_matrix(y_test,y_pred)

array([[808, 144],
       [166, 879]])

In [105]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 45.3 MB/s eta 0:00:00


In [106]:
import gensim